# Step 1. Data Injection

Upload csv files and transfer them to DuckDB file as **`Raw Schema`**.

Load all csv files into DuctDB tables

In [ ]:
import pandas as pd
import duckdb
import os

# Initialize DuckDB in-memory database
con = duckdb.connect(database=':memory:', read_only=False)

# Define the directory where CSV files are located
csv_dir = '/content/'

# Get a list of all files in the directory
all_files = os.listdir(csv_dir)

# Filter to include only CSV files
csv_files = [f for f in all_files if f.endswith('.csv')]

print(f"Found {len(csv_files)} CSV files in '{csv_dir}'.")

# Process each CSV file
for file_name in csv_files:
    full_path = os.path.join(csv_dir, file_name)
    try:
        # Read the CSV file into a pandas DataFrame, interpreting 'None' as nulls
        df = pd.read_csv(full_path, na_values=['None'])

        # Clean up file name for DuckDB table name (remove extension and replace invalid characters)
        table_name = file_name.rsplit('.', 1)[0].replace(' ', '_').replace('-', '_')

        # Register DataFrame as a DuckDB table
        con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM df")
        print(f"'{file_name}' loaded into DuckDB table: '{table_name}' with 'None' as nulls.")
    except Exception as e:
        print(f"Error processing '{file_name}': {e}")

print("\nAll available CSVs are now loaded as tables in DuckDB (if successful).")

# Verify the data loading
print("\n--- Verification ---")
print("Tables in DuckDB:")
all_tables = con.execute("SHOW TABLES").fetchall()
for table_tuple in all_tables:
    table_name = table_tuple[0]
    print(f"\nTable: {table_name}")
    print("Schema:")
    print(con.execute(f"DESCRIBE {table_name}").fetchdf())
    print("First 5 rows (including potential nulls):")
    print(con.execute(f"SELECT * FROM {table_name} LIMIT 5").fetchdf())
    print("\n")

Based on results, common columns are `customer_id`, `order_id`, `review_id`, `product_id`, `seller_i`, and `order_item_id`.

Create a SQL view named `main_orders_view` by performing LEFT JOINs on

`olist_orders_dataset`

`olist_customers_dataset`

`olist_order_items_dataset`

`olist_order_payments_dataset`

using `order_id` and `customer_id` as common columns, then verify the view creation and its schema.


In [ ]:
sql_query = """CREATE OR REPLACE VIEW main_orders_view AS
SELECT
    o.*,
    c.customer_unique_id,
    c.customer_zip_code_prefix,
    c.customer_city,
    c.customer_state,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.shipping_limit_date,
    oi.price,
    oi.freight_value,
    op.payment_sequential,
    op.payment_type,
    op.payment_installments,
    op.payment_value
FROM olist_orders_dataset AS o
LEFT JOIN olist_customers_dataset AS c
    ON o.customer_id = c.customer_id
LEFT JOIN olist_order_items_dataset AS oi
    ON o.order_id = oi.order_id
LEFT JOIN olist_order_payments_dataset AS op
    ON o.order_id = op.order_id;
"""

con.execute(sql_query)
print("View 'main_orders_view' created successfully in DuckDB.")

# Verify the view by listing tables/views and describing it
print("\nTables and Views in DuckDB after creation:")
print(con.execute("SHOW TABLES").fetchall())
print("\nSchema of 'main_orders_view':")
print(con.execute("DESCRIBE main_orders_view").fetchall())

**Summary**

The process of creating the `main_orders_view` involved constructing a SQL query that performed `LEFT JOIN` operations on four datasets: `olist_orders_dataset`

`olist_customers_dataset`

`olist_order_items_dataset`

`olist_order_payments_dataset`.


The joins were made using `order_id` and `customer_id` as common columns.

The view's readiness for further analysis was confirmed by successfully executing the creation query, verifying its presence among database objects, and describing its schema, which accurately reflected the joined columns.



**Data Analysis Key Findings**
*   A SQL view named `main_orders_view` was successfully created in the DuckDB database using a `CREATE OR REPLACE VIEW` statement.
*   The `main_orders_view` was confirmed to be present in the database through the `SHOW TABLES` command.
*   The schema of `main_orders_view` was verified using `DESCRIBE main_orders_view`, confirming that all selected columns from the joined tables (`olist_orders_dataset`, `olist_customers_dataset`, `olist_order_items_dataset`, and `olist_order_payments_dataset`) were correctly included, indicating successful `LEFT JOIN` operations.

**Insights or Next Steps**
*   The `main_orders_view` is now available and properly structured, providing a unified dataset ready for comprehensive order-related analysis.



In [ ]:
# Calculate basic statistics from the main_orders_view
stats_query = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM main_orders_view;
"""

# Fetch the results into a pandas DataFrame
df_stats = con.execute(stats_query).fetchdf()

# Print the results
print("--- Main Orders View Statistics ---")
print(df_stats.to_string(index=False))

# Calculate and print order-to-customer ratio
ratio = df_stats['unique_orders'][0] / df_stats['unique_customers'][0]
print(f"\nAverage orders per unique customer: {ratio:.2f}")

In [ ]:
# check total records in each table
for table_tuple in all_tables:
    # Extract the name from the tuple (usually the first element)
    table_name = table_tuple[0]

    print(f"\n{'='*30}")
    print(f"Table: {table_name}")
    print(f"{'='*30}")

    print("\nSchema:")
    print(con.execute(f"DESCRIBE {table_name}").fetchdf())

    print("\nFirst 5 rows (including potential nulls):")
    # Added the closing parenthesis and .fetchdf() to complete the logic
    print(con.execute(f"SELECT * FROM {table_name} LIMIT 5").fetchdf())

# check total record of each table in all_tables
print("\n--- Total Records in Each Table ---")
# Calculate the max length for pretty alignment (optional but nice)
max_len = max(len(t[0]) for t in all_tables)

for table_tuple in all_tables:
    table_name = table_tuple[0]
    # Fetch the count
    total_records = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

    # Use f-string padding to align the counts
    print(f"{table_name:<{max_len + 2}} : {total_records:,}")

Export the in-memory DuckDB database to a persistent file and download it to the local machine.


In [ ]:
import os
from google.colab import files

# 1. Configuration
db_file = 'olist_ecommerce.db'
alias = 'persistent_db'

# 2. Cleanup: Handle existing attachments or old files
try:
    existing_dbs = con.execute("SELECT database_name FROM duckdb_databases()").fetchall()
    if any(db[0] == alias for db in existing_dbs):
        print(f"Detaching existing {alias}...")
        con.execute(f"DETACH {alias}")
except Exception:
    pass # If we can't even check databases, we'll just try to move on

if os.path.exists(db_file):
    os.remove(db_file)
    print(f"Removed old {db_file} to start fresh.")

# 3. Attach the new database file
con.execute(f"ATTACH '{db_file}' AS {alias}")

# 4. Identify objects to migrate (Legacy-Compatible Version)
# Get Tables
tables = con.execute("SELECT table_name FROM duckdb_tables() WHERE schema_name = 'main'").fetchall()
# Get Views
views = con.execute("SELECT view_name FROM duckdb_views() WHERE schema_name = 'main'").fetchall()

print(f"Found {len(tables)} tables and {len(views)} views to export.")

# 5. Migration Loop - Tables
for (table_name,) in tables:
    try:
        con.execute(f"CREATE TABLE {alias}.{table_name} AS SELECT * FROM main.{table_name}")
        print(f"  [Table] {table_name} copied.")
    except Exception as e:
        print(f"  [Error] Could not copy table {table_name}: {e}")

# 6. Migration Loop - Views
for (view_name,) in views:
    try:
        # Get the SQL definition
        view_sql = con.execute(f"SELECT sql FROM duckdb_views() WHERE view_name = '{view_name}'").fetchone()[0]
        # Recreate view (splitting at 'AS' to get the query part)
        query_part = view_sql.split('AS', 1)[1]
        con.execute(f"CREATE VIEW {alias}.{view_name} AS {query_part}")
        print(f"  [View]  {view_name} recreated.")
    except Exception as e:
        print(f"  [Error] Could not recreate view {view_name}: {e}")

# 7. Finalize and Download
con.execute(f"DETACH {alias}")
print(f"\n✅ Database successfully exported to {db_file}")

files.download(db_file)

# Step 2. Data Warehouse Design

Create a **`Star Schema`** in the Same DuctDB Data File.

Since this step 2 may be not run in the same session as Step 1, the db_file will be reloaded if necessary.

In [ ]:
import duckdb
import os
from google.colab import files

db_file = 'olist_ecommerce.db'

# Check if the database file already exists in the Colab environment
if os.path.exists(db_file):
    print(f"'{db_file}' found. Connecting to existing database.")
    con = duckdb.connect(database=db_file, read_only=False)
else:
    print(f"'{db_file}' not found. Please upload the DuckDB file from your local machine.")
    print("If you previously exported it, it would have been downloaded to your computer.")
    uploaded = files.upload() # This will prompt user to upload.

    if db_file not in uploaded:
        print(f"Error: '{db_file}' was not uploaded. Please ensure you upload the correct file.")
        # Raise an error to stop execution if the required file isn't uploaded
        raise FileNotFoundError(f"'{db_file}' was not uploaded. Please restart the cell and upload the file.")
    else:
        print(f"'{db_file}' uploaded successfully. Connecting to database.")
        con = duckdb.connect(database=db_file, read_only=False)

print(f"Successfully connected to DuckDB database: {db_file}")

# Verify tables in the loaded database
print("\nTables in the loaded DuckDB database:")
print(con.execute("SHOW TABLES").fetchdf())

In [ ]:
# create a 'Star Schema' called 'analytics' if not exist in db_file

# Create the 'analytics' schema if it doesn't already exist
con.execute("CREATE SCHEMA IF NOT EXISTS analytics;")
print("Schema 'analytics' created or already exists.")

# Verify the creation by listing schemas using the correct DuckDB system view
print("\nSchemas in DuckDB:")
print(con.execute("SELECT * FROM duckdb_schemas();").fetchdf())

Further steps:

**Create Dimension Tables (DDL)**:
    *   Generate and execute SQL DDL statements to create the following tables in the `analytics` schema, including surrogate keys for each:
        *   `analytics.dim_customer`
        *   `analytics.dim_product`
        *   `analytics.dim_seller`
        *   `analytics.dim_geolocation`
        *   `analytics.dim_time`


Examine the schemas of `main_orders_view`, `olist_products_dataset`, `product_category_name_translation`, `olist_sellers_dataset`, and `olist_geolocation_dataset` to identify appropriate attributes for each dimension and fact table.


In [ ]:
print("\n--- Schema of main_orders_view ---")
print(con.execute("DESCRIBE main_orders_view").fetchdf())

print("\n--- Schema of olist_products_dataset ---")
print(con.execute("DESCRIBE olist_products_dataset").fetchdf())

print("\n--- Schema of product_category_name_translation ---")
print(con.execute("DESCRIBE product_category_name_translation").fetchdf())

print("\n--- Schema of olist_sellers_dataset ---")
print(con.execute("DESCRIBE olist_sellers_dataset").fetchdf())

print("\n--- Schema of olist_geolocation_dataset ---")
print(con.execute("DESCRIBE olist_geolocation_dataset").fetchdf())


Based on the schema analysis of `main_orders_view`, `olist_products_dataset`, `product_category_name_translation`, `olist_sellers_dataset`, and `olist_geolocation_dataset`, the following dimension and fact tables are proposed for the `analytics` schema:

 1. Dimension Table: `analytics.dim_customer`
This table will store unique customer information.
- `customer_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the customer dimension.
- `customer_id` (VARCHAR): Natural key from `olist_customers_dataset` and `main_orders_view`.
- `customer_unique_id` (VARCHAR): Unique identifier for a customer from `main_orders_view`.
- `customer_zip_code_prefix` (BIGINT): Customer's zip code prefix from `main_orders_view`.
- `customer_city` (VARCHAR): Customer's city from `main_orders_view`.
- `customer_state` (VARCHAR): Customer's state from `main_orders_view`.

 2. Dimension Table: `analytics.dim_product`
This table will store product details.
- `product_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the product dimension.
- `product_id` (VARCHAR): Natural key from `olist_products_dataset` and `main_orders_view`.
- `product_category_name` (VARCHAR): Original product category name from `olist_products_dataset`.
- `product_category_name_english` (VARCHAR): Translated product category name from `product_category_name_translation`.
- `product_name_lenght` (DOUBLE): Product name length from `olist_products_dataset`.
- `product_description_lenght` (DOUBLE): Product description length from `olist_products_dataset`.
- `product_photos_qty` (DOUBLE): Number of product photos from `olist_products_dataset`.
- `product_weight_g` (DOUBLE): Product weight in grams from `olist_products_dataset`.
- `product_length_cm` (DOUBLE): Product length in cm from `olist_products_dataset`.
- `product_height_cm` (DOUBLE): Product height in cm from `olist_products_dataset`.
- `product_width_cm` (DOUBLE): Product width in cm from `olist_products_dataset`.

 3. Dimension Table: `analytics.dim_seller`
This table will store seller information.
- `seller_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the seller dimension.
- `seller_id` (VARCHAR): Natural key from `olist_sellers_dataset` and `main_orders_view`.
- `seller_zip_code_prefix` (BIGINT): Seller's zip code prefix from `olist_sellers_dataset`.
- `seller_city` (VARCHAR): Seller's city from `olist_sellers_dataset`.
- `seller_state` (VARCHAR): Seller's state from `olist_sellers_dataset`.

 4. Dimension Table: `analytics.dim_geolocation`
This table will store geographical data, primarily based on zip code prefixes.
- `geolocation_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the geolocation dimension.
- `geolocation_zip_code_prefix` (BIGINT): Zip code prefix from `olist_geolocation_dataset`.
- `geolocation_lat` (DOUBLE): Latitude from `olist_geolocation_dataset`.
- `geolocation_lng` (DOUBLE): Longitude from `olist_geolocation_dataset`.
- `geolocation_city` (VARCHAR): City name from `olist_geolocation_dataset`.
- `geolocation_state` (VARCHAR): State name from `olist_geolocation_dataset`.

 5. Dimension Table: `analytics.dim_time`
This table will store date and time components derived from order timestamps.
- `time_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the time dimension (YYYYMMDD).
- `full_date` (DATE): The full date.
- `year` (INTEGER): Year component.
- `month` (INTEGER): Month component (1-12).
- `day` (INTEGER): Day component (1-31).
- `quarter` (INTEGER): Quarter of the year (1-4).
- `day_of_week` (INTEGER): Day of the week (1=Sunday, 7=Saturday).
- `day_name` (VARCHAR): Name of the day (e.g., 'Monday').
- `month_name` (VARCHAR): Name of the month (e.g., 'January').
- `week_of_year` (INTEGER): Week number of the year.
- `is_weekend` (BOOLEAN): Flag indicating if the day is a weekend.

 6. Fact Table: `analytics.fact_orders`
This table will store the core transactional data and measures.
- `order_pk` (BIGINT, PRIMARY KEY, AUTOINCREMENT): Surrogate key for the fact table.
- `order_id` (VARCHAR): Natural key from `main_orders_view`.
- `customer_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_customer.customer_pk`.
- `product_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_product.product_pk`.
- `seller_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_seller.seller_pk`.
- `order_purchase_date_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_time.time_pk` for order purchase date.
- `order_approved_date_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_time.time_pk` for order approved date.
- `order_delivered_carrier_date_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_time.time_pk` for carrier delivery date.
- `order_delivered_customer_date_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_time.time_pk` for customer delivery date.
- `order_estimated_delivery_date_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_time.time_pk` for estimated delivery date.
- `customer_zip_geolocation_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_geolocation.geolocation_pk` for customer location.
- `seller_zip_geolocation_fk` (BIGINT, FOREIGN KEY): References `analytics.dim_geolocation.geolocation_pk` for seller location.
- `order_status` (VARCHAR): Current status of the order.
- `order_item_id` (BIGINT): Identifier for each item within an order (from `main_orders_view`).
- `shipping_limit_date` (VARCHAR): The date when the shipping limit was met (from `main_orders_view`).
- `price` (DOUBLE): Price of the product item (from `main_orders_view`).
- `freight_value` (DOUBLE): Freight value for the item (from `main_orders_view`).
- `payment_sequential` (BIGINT): Sequential number of a payment means for a given order (from `main_orders_view`).
- `payment_type` (VARCHAR): Type of payment used (from `main_orders_view`).
- `payment_installments` (BIGINT): Number of installments for the payment (from `main_orders_view`).
- `payment_value` (DOUBLE): Total value of the payment (from `main_orders_view`).


In [ ]:
print("Creating dimension tables in 'analytics' schema...")

# Drop dependent fact table first if it exists to resolve dependency issues
con.execute("DROP TABLE IF EXISTS analytics.fact_orders;")
print("Dropped analytics.fact_orders (if it existed) to resolve dependencies.")

# 1. Create analytics.dim_customer
con.execute("""CREATE OR REPLACE TABLE analytics.dim_customer (
    customer_pk BIGINT PRIMARY KEY,
    customer_id VARCHAR,
    customer_unique_id VARCHAR,
    customer_zip_code_prefix BIGINT,
    customer_city VARCHAR,
    customer_state VARCHAR
);
""")
print("Created analytics.dim_customer.")

# 2. Create analytics.dim_product
con.execute("""CREATE OR REPLACE TABLE analytics.dim_product (
    product_pk BIGINT PRIMARY KEY,
    product_id VARCHAR,
    product_category_name VARCHAR,
    product_category_name_english VARCHAR,
    product_name_lenght DOUBLE,
    product_description_lenght DOUBLE,
    product_photos_qty DOUBLE,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE
);
""")
print("Created analytics.dim_product.")

# 3. Create analytics.dim_seller
con.execute("""CREATE OR REPLACE TABLE analytics.dim_seller (
    seller_pk BIGINT PRIMARY KEY,
    seller_id VARCHAR,
    seller_zip_code_prefix BIGINT,
    seller_city VARCHAR,
    seller_state VARCHAR
);
""")
print("Created analytics.dim_seller.")

# 4. Create analytics.dim_geolocation
con.execute("""CREATE OR REPLACE TABLE analytics.dim_geolocation (
    geolocation_pk BIGINT PRIMARY KEY,
    geolocation_zip_code_prefix BIGINT,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    geolocation_city VARCHAR,
    geolocation_state VARCHAR
);
""")
print("Created analytics.dim_geolocation.")

# 5. Create analytics.dim_time
# Note: This dim_time table creation will be more complex as it needs to generate dates.
# For now, create a basic structure. Population will be done later.
con.execute("""CREATE OR REPLACE TABLE analytics.dim_time (
    time_pk INTEGER PRIMARY KEY,
    full_date DATE,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    day_of_week INTEGER,
    day_name VARCHAR,
    month_name VARCHAR,
    week_of_year INTEGER,
    is_weekend BOOLEAN
);
""")
print("Created analytics.dim_time.")

print("All dimension tables DDL executed.")

In [ ]:
print("Dropping and recreating dimension tables in 'analytics' schema...")

# Drop existing tables
con.execute("DROP TABLE IF EXISTS analytics.dim_customer;")
con.execute("DROP TABLE IF EXISTS analytics.dim_product;")
con.execute("DROP TABLE IF EXISTS analytics.dim_seller;")
con.execute("DROP TABLE IF EXISTS analytics.dim_geolocation;")
con.execute("DROP TABLE IF EXISTS analytics.dim_time;")
print("Existing dimension tables dropped.")

# 1. Recreate analytics.dim_customer
con.execute("""CREATE OR REPLACE TABLE analytics.dim_customer (
    customer_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    customer_id VARCHAR,
    customer_unique_id VARCHAR,
    customer_zip_code_prefix BIGINT,
    customer_city VARCHAR,
    customer_state VARCHAR
);
""")
print("Recreated analytics.dim_customer with BIGINT PRIMARY KEY.")

# 2. Recreate analytics.dim_product
con.execute("""CREATE OR REPLACE TABLE analytics.dim_product (
    product_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    product_id VARCHAR,
    product_category_name VARCHAR,
    product_category_name_english VARCHAR,
    product_name_lenght DOUBLE,
    product_description_lenght DOUBLE,
    product_photos_qty DOUBLE,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE
);
""")
print("Recreated analytics.dim_product with BIGINT PRIMARY KEY.")

# 3. Recreate analytics.dim_seller
con.execute("""CREATE OR REPLACE TABLE analytics.dim_seller (
    seller_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    seller_id VARCHAR,
    seller_zip_code_prefix BIGINT,
    seller_city VARCHAR,
    seller_state VARCHAR
);
""")
print("Recreated analytics.dim_seller with BIGINT PRIMARY KEY.")

# 4. Recreate analytics.dim_geolocation
con.execute("""CREATE OR REPLACE TABLE analytics.dim_geolocation (
    geolocation_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    geolocation_zip_code_prefix BIGINT,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    geolocation_city VARCHAR,
    geolocation_state VARCHAR
);
""")
print("Recreated analytics.dim_geolocation with BIGINT PRIMARY KEY.")

# 5. Recreate analytics.dim_time
con.execute("""CREATE OR REPLACE TABLE analytics.dim_time (
    time_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    full_date DATE,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    day_of_week INTEGER,
    day_name VARCHAR,
    month_name VARCHAR,
    week_of_year INTEGER,
    is_weekend BOOLEAN
);
""")
print("Recreated analytics.dim_time with BIGINT PRIMARY KEY.")

print("All dimension tables dropped and recreated with BIGINT primary keys.")

**Populate Dimension Tables (DML)**:
    Generate and execute SQL INSERT statements to populate each dimension table by selecting distinct values and relevant attributes from the source tables and `main_orders_view`. Assign surrogate keys during this process.

In [ ]:
print("Populating analytics.dim_customer...")

con.execute("""INSERT INTO analytics.dim_customer (customer_pk, customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state)
SELECT
    ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_pk,
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM (
    SELECT DISTINCT
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state
    FROM main_orders_view
) AS distinct_customers;
""")
print("analytics.dim_customer populated successfully.")

# Verify analytics.dim_customer
print("\n--- Verification of analytics.dim_customer ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.dim_customer").fetchdf())

print("\nFirst 5 rows:")
print(con.execute("SELECT * FROM analytics.dim_customer LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.dim_customer").fetchdf())

In [ ]:
print("Populating analytics.dim_product...")

con.execute("""INSERT INTO analytics.dim_product (
    product_pk,
    product_id,
    product_category_name,
    product_category_name_english,
    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
)
SELECT
    ROW_NUMBER() OVER (ORDER BY op.product_id) AS product_pk,
    op.product_id,
    op.product_category_name,
    pcnt.product_category_name_english,
    op.product_name_lenght,
    op.product_description_lenght,
    op.product_photos_qty,
    op.product_weight_g,
    op.product_length_cm,
    op.product_height_cm,
    op.product_width_cm
FROM (
    SELECT DISTINCT
        product_id,
        product_category_name,
        product_name_lenght,
        product_description_lenght,
        product_photos_qty,
        product_weight_g,
        product_length_cm,
        product_height_cm,
        product_width_cm
    FROM olist_products_dataset
) AS op
LEFT JOIN product_category_name_translation AS pcnt
    ON op.product_category_name = pcnt.product_category_name;
""")
print("analytics.dim_product populated successfully.")

# Verify analytics.dim_product
print("\n--- Verification of analytics.dim_product ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.dim_product").fetchdf())

print("\nFirst 5 rows:")
print(con.execute("SELECT * FROM analytics.dim_product LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.dim_product").fetchdf())

In [ ]:
print("Populating analytics.dim_seller...")

con.execute("""INSERT INTO analytics.dim_seller (
    seller_pk,
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
)
SELECT
    ROW_NUMBER() OVER (ORDER BY seller_id) AS seller_pk,
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
FROM (
    SELECT DISTINCT
        seller_id,
        seller_zip_code_prefix,
        seller_city,
        seller_state
    FROM olist_sellers_dataset
) AS distinct_sellers;
""")
print("analytics.dim_seller populated successfully.")

# Verify analytics.dim_seller
print("\n--- Verification of analytics.dim_seller ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.dim_seller").fetchdf())

print("\nFirst 5 rows:")
print(con.execute("SELECT * FROM analytics.dim_seller LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.dim_seller").fetchdf())

In [ ]:
print("Populating analytics.dim_geolocation...")

con.execute("DELETE FROM analytics.dim_geolocation;")
print("Existing data in analytics.dim_geolocation deleted.")

# Insert a default 'Unknown' geolocation entry with geolocation_pk = 0
con.execute("""
INSERT INTO analytics.dim_geolocation (geolocation_pk, geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state)
VALUES (0, NULL, NULL, NULL, 'Unknown', 'Unknown');
""")
print("Inserted 'Unknown' geolocation entry into analytics.dim_geolocation.")

con.execute("""INSERT INTO analytics.dim_geolocation (
    geolocation_pk,
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
)
SELECT
    ROW_NUMBER() OVER (ORDER BY geolocation_zip_code_prefix, geolocation_lat, geolocation_lng) AS geolocation_pk,
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
FROM (
    SELECT DISTINCT
        geolocation_zip_code_prefix,
        geolocation_lat,
        geolocation_lng,
        geolocation_city,
        geolocation_state
    FROM olist_geolocation_dataset
) AS distinct_geolocations
WHERE geolocation_zip_code_prefix IS NOT NULL; -- Exclude entries where zip_code_prefix is NULL, as our unknown entry handles this
""")
print("analytics.dim_geolocation populated successfully with generated geolocations.")

# Verify analytics.dim_geolocation
print("\n--- Verification of analytics.dim_geolocation ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.dim_geolocation").fetchdf())

print("\nFirst 5 rows (including Unknown if present):")
print(con.execute("SELECT * FROM analytics.dim_geolocation ORDER BY geolocation_pk LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.dim_geolocation").fetchdf())


In [ ]:
print("Determining date range for dim_time...")

date_range_query = """SELECT
    MIN(date_val) AS min_date,
    MAX(date_val) AS max_date
FROM (
    SELECT CAST(order_purchase_timestamp AS DATE) AS date_val FROM main_orders_view
    UNION ALL
    SELECT CAST(order_approved_at AS DATE) FROM main_orders_view
    UNION ALL
    SELECT CAST(order_delivered_carrier_date AS DATE) FROM main_orders_view
    UNION ALL
    SELECT CAST(order_delivered_customer_date AS DATE) FROM main_orders_view
    UNION ALL
    SELECT CAST(order_estimated_delivery_date AS DATE) FROM main_orders_view
) AS all_dates
WHERE date_val IS NOT NULL;
"""

date_range = con.execute(date_range_query).fetchdf()

min_date = date_range['min_date'][0]
max_date = date_range['max_date'][0]

print(f"Minimum overall date: {min_date}")
print(f"Maximum overall date: {max_date}")

In [ ]:
print("Populating analytics.dim_time...")

con.execute("DELETE FROM analytics.dim_time;")
print("Existing data in analytics.dim_time deleted.")

# Insert a default 'Unknown' date entry
con.execute("""
INSERT INTO analytics.dim_time (time_pk, full_date, year, month, day, quarter, day_of_week, day_name, month_name, week_of_year, is_weekend)
VALUES (0, NULL, NULL, NULL, NULL, NULL, NULL, 'Unknown', 'Unknown', NULL, NULL);
""")
print("Inserted 'Unknown' date entry into analytics.dim_time.")

con.execute(f"""INSERT INTO analytics.dim_time (
    time_pk,
    full_date,
    year,
    month,
    day,
    quarter,
    day_of_week,
    day_name,
    month_name,
    week_of_year,
    is_weekend
)
SELECT
    CAST(STRFTIME(date_series, '%Y%m%d') AS BIGINT) AS time_pk,
    date_series AS full_date,
    EXTRACT(YEAR FROM date_series) AS year,
    EXTRACT(MONTH FROM date_series) AS month,
    EXTRACT(DAY FROM date_series) AS day,
    EXTRACT(QUARTER FROM date_series) AS quarter,
    (EXTRACT(DOW FROM date_series) + 1) AS day_of_week, -- Adjust to 1=Sunday, 7=Saturday
    STRFTIME(date_series, '%A') AS day_name, -- Use %A for full weekday name (e.g., 'Sunday')
    STRFTIME(date_series, '%B') AS month_name, -- Use %B for full month name (e.g., 'January')
    EXTRACT(WEEK FROM date_series) AS week_of_year,
    CASE WHEN EXTRACT(DOW FROM date_series) IN (0, 6) THEN TRUE ELSE FALSE END AS is_weekend
FROM (
    SELECT UNNEST(GENERATE_SERIES('{min_date}'::DATE, '{max_date}'::DATE, INTERVAL 1 DAY)) AS date_series
) AS dates;
""")
print("analytics.dim_time populated successfully with generated dates.")

# Verify analytics.dim_time
print("\n--- Verification of analytics.dim_time ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.dim_time").fetchdf())

print("\nFirst 5 rows (including Unknown if present):")
print(con.execute("SELECT * FROM analytics.dim_time ORDER BY time_pk LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.dim_time").fetchdf())



**Create Fact Table (DDL)**:
    Generate and execute SQL DDL statements to create `analytics.fact_orders`, including foreign key columns referencing the surrogate keys of the dimension tables and all identified measures (facts).


In [ ]:
print("Creating analytics.fact_orders table...")

# if analytics.fact_order table exists, drop it first
# con.execute("DROP TABLE IF EXISTS analytics.fact_orders;")

con.execute("""CREATE OR REPLACE TABLE analytics.fact_orders (
    order_pk BIGINT PRIMARY KEY, -- Removed GENERATED ALWAYS AS IDENTITY
    order_id VARCHAR,
    customer_fk BIGINT,
    product_fk BIGINT,
    seller_fk BIGINT,
    order_purchase_date_fk BIGINT,
    order_approved_date_fk BIGINT,
    order_delivered_carrier_date_fk BIGINT,
    order_delivered_customer_date_fk BIGINT,
    order_estimated_delivery_date_fk BIGINT,
    customer_zip_geolocation_fk BIGINT,
    seller_zip_geolocation_fk BIGINT,
    order_status VARCHAR,
    order_item_id BIGINT,
    shipping_limit_date VARCHAR,
    price DOUBLE,
    freight_value DOUBLE,
    payment_sequential BIGINT,
    payment_type VARCHAR,
    payment_installments BIGINT,
    payment_value DOUBLE
);
""")

print("analytics.fact_orders table created successfully.")


**Populate Fact Table (DML)**:
    Generate and execute SQL INSERT statements to populate `analytics.fact_orders`, joining `main_orders_view` with the newly created dimension tables to correctly map natural keys to surrogate keys and insert the factual data.


In [ ]:
print("Populating analytics.fact_orders...")

# Clear existing data to avoid primary key constraint violations when re-running
con.execute("DELETE FROM analytics.fact_orders;")
print("Existing data in analytics.fact_orders deleted.")

# --- ADDED VERIFICATION STEP ---
count_after_delete = con.execute("SELECT COUNT(*) FROM analytics.fact_orders;").fetchone()[0]
print(f"Count of analytics.fact_orders after DELETE: {count_after_delete}")
# --- END ADDED VERIFICATION STEP ---

con.execute("""INSERT INTO analytics.fact_orders (
    order_pk,
    order_id,
    customer_fk,
    product_fk,
    seller_fk,
    order_purchase_date_fk,
    order_approved_date_fk,
    order_delivered_carrier_date_fk,
    order_delivered_customer_date_fk,
    order_estimated_delivery_date_fk,
    customer_zip_geolocation_fk,
    seller_zip_geolocation_fk,
    order_status,
    order_item_id,
    shipping_limit_date,
    price,
    freight_value,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
)
SELECT
    ('0x' || SUBSTRING(MD5(GEN_RANDOM_UUID()::VARCHAR), 1, 15))::BIGINT AS order_pk, -- Generate unique BIGINT from MD5 hash of UUID (with explicit cast)
    mov.order_id,
    COALESCE(dc.customer_pk, 0) AS customer_fk,
    COALESCE(dp.product_pk, 0) AS product_fk,
    COALESCE(ds.seller_pk, 0) AS seller_fk,
    COALESCE(CAST(STRFTIME(TRY_CAST(mov.order_purchase_timestamp AS DATE), '%Y%m%d') AS BIGINT), 0) AS order_purchase_date_fk,
    COALESCE(CAST(STRFTIME(TRY_CAST(mov.order_approved_at AS DATE), '%Y%m%d') AS BIGINT), 0) AS order_approved_date_fk,
    COALESCE(CAST(STRFTIME(TRY_CAST(mov.order_delivered_carrier_date AS DATE), '%Y%m%d') AS BIGINT), 0) AS order_delivered_carrier_date_fk,
    COALESCE(CAST(STRFTIME(TRY_CAST(mov.order_delivered_customer_date AS DATE), '%Y%m%d') AS BIGINT), 0) AS order_delivered_customer_date_fk,
    COALESCE(CAST(STRFTIME(TRY_CAST(mov.order_estimated_delivery_date AS DATE), '%Y%m%d') AS BIGINT), 0) AS order_estimated_delivery_date_fk,
    COALESCE(customer_geo_lookup.geolocation_pk, 0) AS customer_zip_geolocation_fk,
    COALESCE(seller_geo_lookup.geolocation_pk, 0) AS seller_zip_geolocation_fk,
    mov.order_status,
    mov.order_item_id,
    mov.shipping_limit_date,
    mov.price,
    mov.freight_value,
    mov.payment_sequential,
    mov.payment_type,
    mov.payment_installments,
    mov.payment_value
FROM main_orders_view AS mov
LEFT JOIN analytics.dim_customer AS dc
    ON mov.customer_id = dc.customer_id AND mov.customer_unique_id = dc.customer_unique_id
LEFT JOIN analytics.dim_product AS dp
    ON mov.product_id = dp.product_id
LEFT JOIN analytics.dim_seller AS ds
    ON mov.seller_id = ds.seller_id
LEFT JOIN (
    SELECT
        geolocation_zip_code_prefix,
        MIN(geolocation_pk) AS geolocation_pk
    FROM analytics.dim_geolocation
    GROUP BY geolocation_zip_code_prefix
) AS customer_geo_lookup
    ON mov.customer_zip_code_prefix = customer_geo_lookup.geolocation_zip_code_prefix
LEFT JOIN (
    SELECT
        geolocation_zip_code_prefix,
        MIN(geolocation_pk) AS geolocation_pk
    FROM analytics.dim_geolocation
    GROUP BY geolocation_zip_code_prefix
) AS seller_geo_lookup
    ON ds.seller_zip_code_prefix = seller_geo_lookup.geolocation_zip_code_prefix;
""")
print("analytics.fact_orders populated successfully.")

# Verify analytics.fact_orders
print("\n--- Verification of analytics.fact_orders ---")
print("Schema:")
print(con.execute("DESCRIBE analytics.fact_orders").fetchdf())

print("\nFirst 5 rows:")
print(con.execute("SELECT * FROM analytics.fact_orders LIMIT 5").fetchdf())

print("\nTotal record count:")
print(con.execute("SELECT COUNT(*) FROM analytics.fact_orders").fetchdf())


**Verify Star Schema Creation**:
    Verify the successful creation and population of all dimension and fact tables by listing tables within the `analytics` schema.
    Display the schema and a sample of data for each created table to confirm correct structure and content.




In [ ]:
print("\n--- Verifying Star Schema Creation ---")

# 1. List all tables within the 'analytics' schema
print("\nTables in 'analytics' schema:")
analytics_tables_df = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'analytics';").fetchdf()
print(analytics_tables_df)

# 2. For each table, display its schema and first 5 rows
if not analytics_tables_df.empty:
    for table_name in analytics_tables_df['table_name']:
        print(f"\n--- Verification for analytics.{table_name} ---")
        print("Schema:")
        print(con.execute(f"DESCRIBE analytics.{table_name}").fetchdf())

        print("First 5 rows:")
        print(con.execute(f"SELECT * FROM analytics.{table_name} LIMIT 5").fetchdf())
else:
    print("No tables found in the 'analytics' schema.")

In [ ]:
print("Parsing and structuring schema details...")

schema_info = {}

# Get the list of tables in the 'analytics' schema from the previous output
# analytics_tables_df is available from the previous cell's execution context
table_names = analytics_tables_df['table_name'].tolist()

for table_name in table_names:
    # Execute DESCRIBE query for the current table
    table_schema_df = con.execute(f"DESCRIBE analytics.{table_name}").fetchdf()

    columns = []
    primary_key = []
    for index, row in table_schema_df.iterrows():
        columns.append({
            'column_name': row['column_name'],
            'column_type': row['column_type']
        })
        if row['key'] == 'PRI':
            primary_key.append(row['column_name'])

    foreign_keys = []
    if table_name == 'fact_orders':
        # Manually identify foreign key relationships from the CREATE TABLE statement (cell c1b31614)
        foreign_keys = [
            {'column_name': 'customer_fk', 'references_table': 'analytics.dim_customer', 'references_column': 'customer_pk'},
            {'column_name': 'product_fk', 'references_table': 'analytics.dim_product', 'references_column': 'product_pk'},
            {'column_name': 'seller_fk', 'references_table': 'analytics.dim_seller', 'references_column': 'seller_pk'},
            {'column_name': 'order_purchase_date_fk', 'references_table': 'analytics.dim_time', 'references_column': 'time_pk'},
            {'column_name': 'order_approved_date_fk', 'references_table': 'analytics.dim_time', 'references_column': 'time_pk'},
            {'column_name': 'order_delivered_carrier_date_fk', 'references_table': 'analytics.dim_time', 'references_column': 'time_pk'},
            {'column_name': 'order_delivered_customer_date_fk', 'references_table': 'analytics.dim_time', 'references_column': 'time_pk'},
            {'column_name': 'order_estimated_delivery_date_fk', 'references_table': 'analytics.dim_time', 'references_column': 'time_pk'},
            {'column_name': 'customer_zip_geolocation_fk', 'references_table': 'analytics.dim_geolocation', 'references_column': 'geolocation_pk'},
            {'column_name': 'seller_zip_geolocation_fk', 'references_table': 'analytics.dim_geolocation', 'references_column': 'geolocation_pk'}
        ]

    schema_info[table_name] = {
        'table_name': table_name,
        'columns': columns,
        'primary_key': primary_key,
        'foreign_keys': foreign_keys
    }

print("Structured Schema Information:")
import json
print(json.dumps(schema_info, indent=4))


   Summarize the successfully created star schema, including the names of the dimension and fact tables, and confirm their readiness for analytical queries.


In [ ]:
print("Generating visual relationship map using Graphviz...")

import graphviz

# Initialize a directed graph
dot = graphviz.Digraph(comment='Star Schema Relationship Map', graph_attr={'rankdir': 'LR'})

# Iterate through schema_info to create nodes for each table
for table_name, details in schema_info.items():
    # Construct the label as a single HTML-like table
    label = f"<<TABLE BORDER=\"0\" CELLBORDER=\"1\" CELLSPACING=\"0\" ALIGN=\"LEFT\"><TR><TD COLSPAN=\"2\"><B>{table_name}</B></TD></TR>\n"

    # Add primary keys and other columns
    for col in details['columns']:
        col_name = col['column_name']
        col_type = col['column_type']
        if col_name in details['primary_key']:
            label += f"<TR><TD ALIGN=\"LEFT\"><B>PK: {col_name}</B></TD><TD ALIGN=\"LEFT\">{col_type}</TD></TR>\n"
        else:
            label += f"<TR><TD ALIGN=\"LEFT\">{col_name}</TD><TD ALIGN=\"LEFT\">{col_type}</TD></TR>\n"

    label += "</TABLE>>"

    dot.node(table_name, label=label)

# Create edges for foreign keys, specifically from fact_orders to dimension tables
if 'fact_orders' in schema_info:
    fact_table_details = schema_info['fact_orders']
    for fk in fact_table_details['foreign_keys']:
        from_col = fk['column_name']
        to_table = fk['references_table'].split('.')[1] # Extract table name from 'analytics.dim_table'
        to_col = fk['references_column']
        dot.edge(fact_table_details['table_name'], to_table, label=f"{from_col} -> {to_col}")

# Render the graph
dot.render('star_schema_relationship_map', view=True, format='png', cleanup=True)
print("Visual relationship map generated as 'star_schema_relationship_map.png'.")

Disciply a relationship map.



In [ ]:
from IPython.display import Image, display

image_path = 'star_schema_relationship_map.png'

print(f"Displaying the generated star schema relationship map: {image_path}")
display(Image(filename=image_path))

print("\n--- Verification ---")
print("Visually inspect the map above to confirm the following:")
print("1. All dimension tables (dim_customer, dim_geolocation, dim_product, dim_seller, dim_time) are present.")
print("2. The fact table (fact_orders) is present.")
print("3. Primary keys (PK) are clearly marked within each table.")
print("4. Foreign keys (FK) in 'fact_orders' are correctly linked to the primary keys of the respective dimension tables.")


Download the DuctDB file.



In [ ]:
from google.colab import files

db_file = 'olist_ecommerce_star.db'

print(f"Downloading '{db_file}' to your local machine...")
files.download(db_file)

print("Download initiated. Please check your browser's download folder.")

# Step 3. ELT Pipeline (The "Tap")

Tap-DuctDB to **`Big Query`**.

Having tried with Meltano which resulted failer on installing tap-ductdb, some other method, like DuctDB extension, Aparche Arrow, or ingestr could be considered. We choose **`Python-native scripts`**.

In [ ]:
# make a data folder in colab. ask user to select olist_ecommerce_star.db to upload into this folder
!mkdir -p data
from google.colab import files
uploaded = files.upload()


Upload service account Json file to colab.

In [ ]:
from google.colab import files
import os

# Upload the service account JSON key file
print("Please select your service account JSON key file to upload:")
uploaded_key = files.upload()

# Assuming you upload only one key file
if uploaded_key:
    key_file_name = list(uploaded_key.keys())[0]
    # Set the GOOGLE_APPLICATION_CREDENTIALS environment variable
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.join('/content/', key_file_name)
    print(f"Service account key '{key_file_name}' uploaded and GOOGLE_APPLICATION_CREDENTIALS set.")
else:
    print("No service account key file was uploaded.")

# Verify that BigQuery client can pick up the credentials
from google.cloud import bigquery
client = bigquery.Client()
print(f"BigQuery client initialized using project: {client.project}")
print("Ensure this project matches the service account you intend to use.")

In [ ]:
import duckdb
import pandas_gbq
from google.cloud import bigquery
import os

# --- CONFIGURATION ---
DB_PATH = 'olist_ecommerce_star.db' # Corrected path based on previous cell's upload
PROJECT_ID = 'ntu-dsai-6m-2026'
DATASET_ID = 'olist_reporting'
LOCATION = 'US' # Or your specific GCP region

# 1. Authenticate to Google Cloud using GOOGLE_APPLICATION_CREDENTIALS (set by previous cell)
print("Authenticating to Google Cloud using provided service account key...")
# The GOOGLE_APPLICATION_CREDENTIALS environment variable should be set by the previous cell.
# No explicit auth.authenticate_user() is needed here if a service account key is used.

# Test the BigQuery connection after authentication
client = bigquery.Client()
print(f"Authenticated using project: {client.project}")

# Ensure the database file exists
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"DuckDB file not found at {DB_PATH}. Please ensure it was uploaded and named correctly.")

# Close any existing DuckDB connection before opening a new one
if 'con' in locals() and isinstance(con, duckdb.DuckDBPyConnection):
    try:
        con.close()
        print("Existing DuckDB connection closed gracefully.")
    except Exception as e:
        print(f"Failed to close existing DuckDB connection: {e}")

# 2. Connect to DuckDB
print(f"Connecting to DuckDB database: {DB_PATH}")
con = duckdb.connect(database=DB_PATH, read_only=False)

# 3. Identify "Gold" tables (those starting with fact_ or dim_) in the 'analytics' schema
tables_query = """
    SELECT table_name
    FROM duckdb_tables()
    WHERE schema_name = 'analytics'
    AND (table_name LIKE 'fact_%' OR table_name LIKE 'dim_%')
    ORDER BY table_name;
"""
tables_to_upload = [row[0] for row in con.execute(tables_query).fetchall()]

print(f"Found {len(tables_to_upload)} tables to inject from 'analytics' schema: {tables_to_upload}")

# 4. Loop and Inject
for table in tables_to_upload:
    print(f"--- Processing {table} ---")

    # Extract to Dataframe
    df = con.execute(f"SELECT * FROM analytics.{table}").df()

    # Inject to BigQuery
    destination = f"{DATASET_ID}.{table}"
    print(f"Uploading {len(df)} rows to {destination}...")

    pandas_gbq.to_gbq(
        df,
        destination,
        project_id=PROJECT_ID,
        if_exists='replace', # Overwrites existing tables for Step 3 fresh starts
        location=LOCATION,
        bigquery_client=client # Explicitly pass the BigQuery client
    )
    print(f"Successfully injected {table}!")

con.close()
print("\n✅ Step 3 Complete: All Star Schema tables are now in BigQuery.")

# Step 4. Data Quality Testing

**Hybrid Approach:** use `Custom SQL Queries` for deep validation and a `Python-based Test Suite` to automate the reporting.

In [ ]:
# Firstly, compare your local DuckDB counts to your BigQuery counts to ensure upload is a success.
import duckdb
import pandas as pd # Import pandas for DataFrame display

# Get Local Counts from DuckDB (WSL/Local)
local_con = duckdb.connect(DB_PATH)
local_counts = {
    "fact_orders": local_con.execute("SELECT COUNT(*) FROM analytics.fact_orders").fetchone()[0],
    "dim_product": local_con.execute("SELECT COUNT(*) FROM analytics.dim_product").fetchone()[0] # Corrected table name and schema
}

# 2. Define the Volumetry Test Function
def run_volumetry_test(table_name, local_count):
    sql = f"SELECT COUNT(*) FROM `olist_reporting.{table_name}`"
    query_job = client.query(sql)
    bq_count = query_job.to_dataframe().iloc[0, 0]

    diff = bq_count - local_count
    status = "✅ MATCH" if diff == 0 else "❌ MISMATCH"

    return {
        "Table": table_name,
        "Local (DuckDB)": local_count,
        "Cloud (BigQuery)": bq_count,
        "Difference": diff,
        "Status": status
    }

# 3. Execute and Display
volumetry_results = [run_volumetry_test(name, count) for name, count in local_counts.items()]
volumetry_df = pd.DataFrame(volumetry_results)

print("--- 📊 Data Volumetry & Reconciliation Report ---")
display(volumetry_df)

In [ ]:
import pandas as pd


def run_dq_test(test_name, sql, expected_result=0):
    """Runs a SQL query and compares the result to the expected outcome."""
    query_job = client.query(sql)
    results = query_job.to_dataframe()
    actual_result = results.iloc[0, 0]

    status = "✅ PASS" if actual_result == expected_result else "❌ FAIL"
    return {
        "Test Name": test_name,
        "Status": status,
        "Issues Found": actual_result,
        "Expected": expected_result
    }

# --- Define the Tests ---
tests = [
    {
        "name": "Null Check: Order IDs in Fact Orders",
        "sql": "SELECT COUNT(*) FROM `olist_reporting.fact_orders` WHERE order_id IS NULL"
    },
    {
        "name": "Duplicate Check: product_id in dim_product",
        "sql": "SELECT COUNT(*) FROM (SELECT product_id FROM `olist_reporting.dim_product` GROUP BY product_id HAVING COUNT(*) > 1)"
    },
    {
        "name": "Referential Integrity: fact_orders.product_fk to dim_product.product_pk",
        "sql": "SELECT COUNT(*) FROM `olist_reporting.fact_orders` WHERE product_fk IS NULL"
    }
]

# --- Execute and Display ---
results_list = [run_dq_test(t['name'], t['sql']) for t in tests]
report_df = pd.DataFrame(results_list)

print("--- 🛡️ Olist Data Quality Report Card ---")
display(report_df)


Download check report.

In [ ]:
!pip install fpdf

In [ ]:
from fpdf import FPDF
from google.colab import files

pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)

# Create a copy of report_df to modify status for PDF export
report_df_for_pdf = report_df.copy()
report_df_for_pdf['Status'] = report_df_for_pdf['Status'].str.replace('✅ PASS', 'PASS').str.replace('❌ FAIL', 'FAIL')

# Convert DataFrame to string for PDF
df_string = report_df_for_pdf.to_string()

# Add a title
pdf.cell(200, 10, txt="Olist Data Quality Report Card", ln=True, align='C')
pdf.ln(10) # Add some space

# Split the string into lines and add to PDF
for line in df_string.split('\n'):
    pdf.cell(0, 5, txt=line, ln=True)

pdf_file_name = "olist_dq_report.pdf"
pdf.output(pdf_file_name)

print(f"Generated PDF: {pdf_file_name}")
files.download(pdf_file_name)
print("Your PDF download should have started.")

Need to check and repopulate all table. Go through the stpe below to check and prepare repopulatoin.

In [ ]:
# Remediation for Referential Integrity: Insert 'Unknown' product into dim_product

import duckdb
import os

# Ensure DB_PATH is defined (it should be a global variable from previous cells)
# if 'DB_PATH' not in globals():
#    DB_PATH = 'olist_ecommerce_star.db'

# Attempt to close any existing connection if 'con' object is in scope
if 'con' in locals() and isinstance(con, duckdb.DuckDBPyConnection):
    try:
        con.close()
        print("Existing DuckDB connection closed gracefully.")
    except Exception as e:
        # Catch various exceptions if con was already closed or in a bad state
        print(f"Failed to close existing DuckDB connection: {e}")
        # We don't want to fail execution here, just log and continue to reconnect

print(f"Re-establishing DuckDB connection to: {DB_PATH}")
con = duckdb.connect(database=DB_PATH, read_only=False)
print("Connection re-established.")

# 1. Check if 'Unknown' product with product_pk = 0 already exists
check_query = "SELECT COUNT(*) FROM analytics.dim_product WHERE product_pk = 0;"
exists = con.execute(check_query).fetchone()[0]

if exists == 0:
    print("Inserting 'Unknown' product into analytics.dim_product...")
    # Insert the 'Unknown' product with a designated product_pk (e.g., 0)
    # The other columns can be set to NULL or placeholder values
    insert_unknown_product_query = """
    INSERT INTO analytics.dim_product (
        product_pk,
        product_id,
        product_category_name,
        product_category_name_english,
        product_name_lenght,
        product_description_lenght,
        product_photos_qty,
        product_weight_g,
        product_length_cm,
        product_height_cm,
        product_width_cm
    )
    VALUES (
        0, -- Designated PK for 'Unknown'
        'unknown_product', -- Placeholder natural key
        'Unknown',
        'Unknown',
        NULL, NULL, NULL, NULL, NULL, NULL, NULL
    );
    """
    con.execute(insert_unknown_product_query)
    print(" 'Unknown' product inserted successfully.")
else:
    print("'Unknown' product with product_pk = 0 already exists in analytics.dim_product.")

# Ensure the connection is closed after operations
con.close()
print("DuckDB connection in cell djXWYA6RsB8x closed.")

# 3. Explain the fact table's current state and next steps
print("\n--- Explanation ---")
print("During the population of analytics.fact_orders (in cell 6bf3e8bc),")
print("the 'product_fk' column was populated using COALESCE(dp.product_pk, 0).")
print("This means that any order items without a matching product in dim_product")
print("should already have their 'product_fk' set to 0.")
print("By inserting an 'Unknown' product with 'product_pk = 0' into analytics.dim_product,")
print("we ensure that these 'product_fk = 0' values now correctly reference a valid")
print("entry in the dimension table, resolving the referential integrity issue.")

print("\n--- Next Steps ---")
print("Please re-run the Data Quality Test cell (LwBiwWx9m3IX) to confirm that")
print("the 'Referential Integrity: fact_orders.product_fk to dim_product.product_pk' test now passes.")

**Insights or Next Steps**
*   The implemented solution effectively resolved the referential integrity issue within the `olist_reporting` dataset, ensuring data consistency for the `product_fk` relationship.
*   The updated ELT process now robustly handles DuckDB connections, preventing potential conflicts and improving the reliability of data loading into BigQuery.


# Step 5. The Analytics Engine (Colab Implementation)

Use `SQLAlchemy` with the `BigQuery dialect`. To treat the cloud warehouse like a standard database within your Python environment.

In [ ]:
# Install the BigQuery SQLAlchemy dialect
!pip install sqlalchemy-bigquery

import pandas as pd
from sqlalchemy import create_engine

# Configuration
PROJECT_ID
DATASET_ID

# Create the SQLAlchemy Engine
# Format: bigquery://{project_id}/{dataset_id}
engine = create_engine(f"bigquery://{PROJECT_ID}/{DATASET_ID}")

print("✅ Step 5: Connected to BigQuery via SQLAlchemy")

## **KPI 1:** Monthly Sales Trends (MoM Growth)

This analysis identifies if the business is growing or shrinking over time.

In [ ]:
query_sales = f"""
SELECT
    DATE_TRUNC(PARSE_DATE('%Y%m%d', CAST(order_purchase_date_fk AS STRING)), MONTH) as month,
    SUM(payment_value) as total_revenue,
    COUNT(DISTINCT order_id) as total_orders
FROM `{PROJECT_ID}.{DATASET_ID}.fact_orders`
WHERE order_purchase_date_fk != 0
GROUP BY 1
ORDER BY 1
"""

df_sales = pd.read_sql(query_sales, engine)
df_sales['revenue_growth'] = df_sales['total_revenue'].pct_change() * 100

print("--- Monthly Sales Performance ---")
display(df_sales.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure and a primary axes object
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot total_revenue on the primary y-axis
sns.lineplot(x='month', y='total_revenue', data=df_sales, ax=ax1, color='blue', label='Total Revenue')
ax1.set_xlabel('Month')
ax1.set_ylabel('Total Revenue', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a secondary y-axis
ax2 = ax1.twinx()

# Plot total_orders on the secondary y-axis
sns.lineplot(x='month', y='total_orders', data=df_sales, ax=ax2, color='red', label='Total Orders')
ax2.set_ylabel('Total Orders', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Add title
plt.title('Monthly Sales Trends: Total Revenue vs. Total Orders')

# Combine legends from both axes
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper left')

# Display the plot
plt.grid(True)
plt.show()

In [ ]:
from google.colab import files

# Save the plot as a PNG file
chart_file_name = 'monthly_sales_trends.png'
fig.savefig(chart_file_name)
print(f"Chart saved as {chart_file_name}")

# Download the chart
files.download(chart_file_name)
print("Download initiated. Please check your browser's download folder.")

*   **Overall Growth Trend**: Both total revenue and total orders show a general upward trend over the observed period, indicating business growth.
*   **Seasonal Fluctuations**: There appear to be some monthly fluctuations in both revenue and orders, suggesting potential seasonality. For example, there might be dips or peaks at certain times of the year.
*   **Correlation**: The two lines (Total Revenue and Total Orders) generally move in tandem, which is expected as more orders typically lead to higher revenue. This correlation reinforces the health of the sales process.
*   **Growth Acceleration/Deceleration**: Specific periods show steeper inclines or plateaus, which could indicate times of accelerated growth or market saturation/challenges. For instance, the initial months show rapid growth, which then appears to stabilize.


## **KPI 2:** Top-Selling Products (Revenue by Category)

To Joine Fact and Dimension tables to see which categories drive the most profit.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from google.colab import files

# Redefine query_products and df_top_products from cell lRFegsrt437Y to ensure it's available
# Assuming PROJECT_ID, DATASET_ID, and engine are available from prior executions
query_products = f"""
SELECT
    p.product_category_name,
    SUM(f.payment_value) as category_revenue,
    COUNT(f.order_id) as items_sold
FROM `{PROJECT_ID}.{DATASET_ID}.fact_orders` f
JOIN `{PROJECT_ID}.{DATASET_ID}.dim_product` p ON f.product_fk = p.product_pk
GROUP BY 1
ORDER BY category_revenue DESC
LIMIT 10
"""

df_top_products = pd.read_sql(query_products, engine)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(x='product_category_name', y='category_revenue', data=df_top_products, palette='viridis', hue='product_category_name', legend=False, ax=ax)
plt.title('Top 10 Product Categories by Revenue')
plt.xlabel('Product Category Name')
plt.ylabel('Category Revenue')
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.tight_layout()

# Save the plot as a PNG file
chart_file_name = 'top_product_categories.png'
fig.savefig(chart_file_name)
print(f"Chart saved as {chart_file_name}")

# Download the chart
files.download(chart_file_name)
print("Download initiated. Please check your browser's download folder.")

plt.show()

In [ ]:
print("--- Top 10 Product Categories by Revenue (DataFrame) ---")
display(df_top_products)

## **KPI 3:** Customer Segmentation (RFM Lite)

Segment customers by their Frequency (how often they buy) and Monetary value (how much they spend).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

query_segmentation = f"""
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) as frequency,
    SUM(payment_value) as monetary_value
FROM `{PROJECT_ID}.{DATASET_ID}.fact_orders` f
JOIN `{PROJECT_ID}.{DATASET_ID}.dim_customer` c ON f.customer_fk = c.customer_pk
GROUP BY 1
"""

df_rfm = pd.read_sql(query_segmentation, engine)

# Simple Python Logic for Segmentation
def segment_customer(row):
    if row['frequency'] > 1 and row['monetary_value'] > 500:
        return 'VIP / Loyal'
    elif row['frequency'] == 1 and row['monetary_value'] > 200:
        return 'High Value / New'
    else:
        return 'Standard'

df_rfm['segment'] = df_rfm.apply(segment_customer, axis=1)

print("\n--- Customer Segmentation Summary ---")
print(df_rfm['segment'].value_counts())

# --- Plotting the Pie Chart ---
segment_counts = df_rfm['segment'].value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel'))
plt.title('Customer Segmentation Distribution')
plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.

# Save the plot as a PNG file
chart_file_name = 'customer_segmentation_pie_chart.png'
fig.savefig(chart_file_name)
print(f"Chart saved as {chart_file_name}")

# Download the chart
files.download(chart_file_name)
print("Download initiated. Please check your browser's download folder.")

plt.show()

The customer segmentation analysis yielded the following distribution:
*   **Standard Customers**: 72,911 (76.1%)
*   **High Value / New Customers**: 22,558 (23.5%)
*   **VIP / Loyal Customers**: 627 (0.4%)

## Start a New Buisiness

If I want to start a same new business, evaluate the data and provide some suggestions.

**Business Insights for a New E-commerce Venture:**

1.  **Market Potential:** The overall growth trend in sales and orders confirms that the e-commerce market is viable and expanding.
2.  **Product Focus:** There's clear demand and revenue generation in specific product categories. Focusing on these proven areas can reduce risk and accelerate market entry.
3.  **Customer Behavior:** A significant portion of the customer base makes one-off or low-value purchases. This implies either a need for stronger loyalty programs or that these categories cater to infrequent but essential purchases. The small VIP segment shows that customer loyalty is achievable.
4.  **Operational Considerations:** Managing seasonal demand and ensuring efficient logistics for diverse product categories will be key.

**Actionable Suggestions for Starting a New E-commerce Business:**

Based on these insights, here are concrete suggestions for a new e-commerce business:

1.  **Product Strategy - Niche Down within Top Categories:**
    *   **Focus Areas:** Instead of trying to cover all top categories, choose one or two of the highest-performing ones, such as "Cama mesa banho" (home goods/textiles) or "Beleza saude" (health and beauty). These have strong revenue potential.
    *   **Specialization:** Within these categories, aim for a niche. For example, instead of all home goods, focus on sustainable or artisan bedding and bath products. For health & beauty, consider organic skincare or specialized wellness products. This allows for deeper market penetration and differentiation.
    *   **High-Quality Sourcing:** Given the competitive landscape, prioritize high-quality, unique, or ethically sourced products that can command better margins and foster customer loyalty.

2.  **Target Market & Marketing Approach:**
    *   **Initial Focus on "High Value / New" Segment:** Design marketing campaigns to convert the large "High Value / New" customer segment into repeat purchasers. This could involve:
        *   **Post-Purchase Engagement:** Follow-up emails with related product recommendations or exclusive discounts for their next purchase.
        *   **Subscription Models:** For categories like health & beauty or consumables (if applicable to the chosen niche), offer subscription options to boost frequency.
        *   **Bundling:** Create attractive product bundles to increase the average order value.
    *   **Nurture VIPs:** While small, the VIP segment is highly valuable. Implement a loyalty program with exclusive perks, early access to new products, or personalized service to retain these customers.
    *   **Leverage Seasonality:** Plan marketing campaigns around observed seasonal peaks and troughs from the `df_sales` data. Stock up on relevant products and launch targeted promotions during high-demand periods.

3.  **Operational & Growth Strategy:**
    *   **Optimized Supply Chain for Chosen Niche:** Develop a robust supply chain specifically tailored to the chosen product categories, ensuring efficient inventory management and fast shipping.
    *   **Focus on Customer Experience:** Given the dominance of "Standard" customers, a new business has a chance to stand out by providing exceptional customer service, clear product descriptions, and hassle-free returns to encourage repeat business.
    *   **Data-Driven Decisions:** Continuously monitor sales trends and customer behavior using KPIs similar to `df_sales`, `df_top_products`, and `df_rfm`. Adapt product offerings, pricing, and marketing strategies based on real-time data.
    *   **Geographic Expansion (Future):** Once established, analyze the `dim_geolocation` data (customer_city, customer_state, geolocation_zip_code_prefix) to identify regions with high concentrations of potential customers or underserved markets for future expansion.

By strategically leveraging these insights, a new e-commerce business can focus its efforts, mitigate risks, and build a sustainable and profitable operation in the dynamic online retail landscape.

**Formulate Business Suggestions**

1.  Based on the insights derived from the 'Monthly Sales Trends', 'Top-Selling Product Categories', and 'Customer Segmentation' analyses, formulate actionable suggestions for a new e-commerce business.
2.  Develop a **Product Strategy** by recommending specific product categories to focus on (e.g., niche within top-performing categories) and how to differentiate from competitors.
3.  Outline a **Target Market & Marketing Approach**, including strategies to convert 'High Value / New' customers into loyal ones and how to nurture the 'VIP / Loyal' segment. Also, consider how to leverage seasonal trends.
4.  Propose **Operational & Growth Strategies**, such as supply chain optimization for chosen niches, focusing on customer experience, and utilizing data for continuous decision-making and potential geographic expansion.

**Key Insights from Analysis:**

**1. Monthly Sales Trends (`df_sales`):**
*   The platform shows a clear overall growth trend in both total revenue and total orders, indicating a growing market opportunity.
*   There are noticeable monthly fluctuations, suggesting seasonality in e-commerce demand. A new business should anticipate and plan for these peaks and troughs.
*   Revenue and order counts are highly correlated, reinforcing that increased order volume directly translates to higher revenue.

**2. Top-Selling Product Categories (`df_top_products`):**
*   **High-Performing Categories:** 'cama_mesa_banho' (bed_bath_table), 'beleza_saude' (health_beauty), 'informatica_acessorios' (computers_accessories), 'moveis_decoracao' (furniture_decor), and 'relogios_presentes' (watches_gifts) consistently drive the highest revenue.
*   These categories often involve household necessities, personal care, electronics, home aesthetics, and gifts, suggesting sustained consumer demand.

**3. Customer Segmentation (`df_rfm`):**
*   **Standard Customers (76.1%):** The largest segment, likely one-time or low-frequency buyers. They represent a significant portion of the market but may require activation strategies.
*   **High Value / New Customers (23.5%):** These customers have made a high-value first purchase or are new and have shown potential. They are prime candidates for conversion into loyal customers.
*   **VIP / Loyal Customers (0.4%):** A small but highly valuable segment, characterized by high frequency and monetary value. These customers are crucial for sustained growth and advocacy.

---

# Step 6 Pipeline Orchestration (Optional)

On hold.

# Step 7. Documentation and Presentation

1. Document your code, data lineage, and pipeline architecture using tools like DRAW.IO, EXCALIDRAW to illustrate the architecture of your data pipeline system
2. Prepare a report summarizing the technical approach and your findings/insights, including relevant tables/charts/graphs
○ Explain why certain tools were chosen over others…etc

## 1. Summary of the technical approach

**Technical Approach Summary**

This project involved building a robust data pipeline to transform raw e-commerce data into a star schema data warehouse in Google BigQuery, enabling analytical querying and KPI generation. The process was structured into several key stages: Data Injection, Data Warehouse Design, ELT Pipeline, Data Quality Testing, and Analytics Engine.

**1. Data Injection into DuckDB (Raw Schema)**

*   **Objective**: Load raw CSV files into a DuckDB database for initial processing.
*   **Methodology**: Python scripts leveraging the `pandas` library were used to read CSV files and the `duckdb` library to create an in-memory database. Each CSV file was loaded into a separate table, forming the "Raw Schema".
*   **Key Tools**: `pandas`, `duckdb`, `os` module for file system interaction.
*   **Challenges/Considerations**: Handling potential `None` values in CSVs by using `na_values=['None']` during `pd.read_csv`. Dynamic table naming from file names was implemented.

**2. Creation of `main_orders_view`**

*   **Objective**: Consolidate key order-related information from multiple raw tables into a single, unified view for easier subsequent processing.
*   **Methodology**: A SQL `CREATE OR REPLACE VIEW` statement was executed against the in-memory DuckDB database. This view performed `LEFT JOIN` operations across `olist_orders_dataset`, `olist_customers_dataset`, `olist_order_items_dataset`, and `olist_order_payments_dataset` using `order_id` and `customer_id` as common columns.
*   **Key Tools**: DuckDB SQL for view creation and verification.
*   **Outcome**: A denormalized `main_orders_view` was created, providing a comprehensive source for building the data warehouse.

**3. Export to a Persistent DuckDB File (`olist_ecommerce.db`)**

*   **Objective**: Persist the processed data and the `main_orders_view` from the in-memory DuckDB to a physical `.db` file, allowing for continuity across sessions and eventual download.
*   **Methodology**: The in-memory database was attached as an alias (`persistent_db`) to a new physical file (`olist_ecommerce.db`). All tables and views from the `main` schema were then copied or recreated within this persistent database. Finally, the `.db` file was downloaded using `google.colab.files.download`.
*   **Key Tools**: `duckdb` (ATTACH, CREATE TABLE AS, CREATE VIEW), `os`, `google.colab.files`.
*   **Challenges/Considerations**: Ensuring proper detachment and re-attachment of the database to prevent conflicts. Extracting SQL definitions for views was necessary to recreate them in the new database.

**4. Star Schema Design and Population in `analytics` Schema**

*   **Objective**: Design and implement a star schema data warehouse within the DuckDB persistent file (`olist_ecommerce.db`) to optimize analytical querying.
*   **Methodology**:
    *   **Schema Creation**: An `analytics` schema was created (`CREATE SCHEMA IF NOT EXISTS analytics`).
    *   **Dimension Table DDL**: SQL `CREATE OR REPLACE TABLE` statements were used to define five dimension tables: `analytics.dim_customer`, `analytics.dim_product`, `analytics.dim_seller`, `analytics.dim_geolocation`, and `analytics.dim_time`. Surrogate keys (`BIGINT PRIMARY KEY`) were defined for each dimension.
    *   **Dimension Table Population (DML)**: `INSERT INTO ... SELECT DISTINCT` statements were used to populate dimension tables by selecting unique attributes from source tables and `main_orders_view`. Surrogate keys were generated using `ROW_NUMBER() OVER (ORDER BY ...)`. Special handling included inserting an 'Unknown' entry (with `_pk = 0`) into `dim_geolocation` and `dim_time` to manage missing or unmatchable foreign keys later. `dim_time` was populated by generating a series of dates between the min and max dates found in `main_orders_view` and extracting various time components.
    *   **Fact Table DDL**: `analytics.fact_orders` was created with its own surrogate key (`order_pk`), natural key (`order_id`), and foreign keys (`_fk` columns) referencing the primary keys of the dimension tables. It also included all relevant measures (e.g., `price`, `payment_value`).
    *   **Fact Table Population (DML)**: `analytics.fact_orders` was populated by joining `main_orders_view` with the newly created dimension tables. `COALESCE(dim_table.pk, 0)` was crucially used for foreign keys to default to the 'Unknown' entry (product_pk=0, time_pk=0, geolocation_pk=0) if a match was not found, ensuring referential integrity. Surrogate keys for the fact table rows were generated using a combination of `MD5(GEN_RANDOM_UUID()::VARCHAR)` to create unique `BIGINT` values.
*   **Key Tools**: DuckDB SQL (CREATE SCHEMA, CREATE TABLE, INSERT INTO, ROW_NUMBER(), MD5(), GEN_RANDOM_UUID(), COALESCE, STRFTIME, EXTRACT, GENERATE_SERIES).
*   **Challenges/Considerations**: Initially, `GENERATED ALWAYS AS IDENTITY` was attempted for surrogate keys but replaced with `ROW_NUMBER()` due to DuckDB's specific behavior with this clause in certain contexts. Careful casting of date strings to `DATE` type was needed for `dim_time` population and for creating `time_pk` values.

**5. ELT Pipeline to Google BigQuery using `pandas_gbq`**

*   **Objective**: Transfer the structured star schema data from DuckDB to Google BigQuery, making it accessible for cloud-based analytics.
*   **Methodology**:
    *   **Authentication**: Google Cloud authentication was handled by uploading a service account JSON key file and setting the `GOOGLE_APPLICATION_CREDENTIALS` environment variable, which `google.cloud.bigquery` and `pandas_gbq` automatically utilized.
    *   **Data Extraction & Loading**: A Python script connected to the persistent DuckDB file. It identified all tables in the `analytics` schema (fact and dimension tables). For each table, the data was extracted into a `pandas` DataFrame (`con.execute(...).df()`) and then loaded into a corresponding BigQuery table using `pandas_gbq.to_gbq()`. The `if_exists='replace'` option was used for simplicity in this prototyping phase.
*   **Key Tools**: `duckdb`, `pandas`, `pandas_gbq`, `google.cloud.bigquery`, `os`.
*   **Challenges/Considerations**: An initial attempt to use Meltano for `tap-duckdb` proved problematic, leading to the decision to use Python-native scripts with `pandas_gbq` for a more controlled and direct ELT process.

**6. Data Quality Testing**

*   **Objective**: Validate the integrity and consistency of the data loaded into BigQuery.
*   **Methodology**: A hybrid approach was employed:
    *   **Volumetry Check**: Compared record counts for key tables (`fact_orders`, `dim_product`) between the source DuckDB and the target BigQuery to ensure all data was transferred successfully.
    *   **Custom SQL Queries (Python-driven)**: Python functions executed BigQuery SQL queries to perform specific data quality checks:
        *   **Null Checks**: Verified that critical columns (e.g., `order_id` in `fact_orders`) did not contain unexpected NULLs.
        *   **Duplicate Checks**: Ensured uniqueness constraints (e.g., `product_id` in `dim_product`) were upheld.
        *   **Referential Integrity Checks**: Verified that foreign key relationships in `fact_orders` correctly referenced corresponding primary keys in dimension tables.
    *   **Remediation**: An issue was identified where `product_fk` in `fact_orders` could be `0` (referencing an 'Unknown' product) but this 'Unknown' product was not guaranteed to exist in `dim_product`. The remediation involved explicitly inserting an 'Unknown' product with `product_pk = 0` into `analytics.dim_product` in DuckDB before the ELT process, ensuring referential integrity was maintained.
    *   **Reporting**: Results were presented in `pandas` DataFrames and exported as a PDF report using `FPDF`.
*   **Key Tools**: `duckdb`, `google.cloud.bigquery`, `pandas`, `FPDF`.

**7. Analytics Engine (SQLAlchemy & KPIs)**

*   **Objective**: Connect to the BigQuery data warehouse and generate key performance indicators (KPIs) with visualizations to derive business insights.
*   **Methodology**:
    *   **Connection**: `SQLAlchemy` with the `sqlalchemy-bigquery` dialect was used to establish a programmatic connection to BigQuery, allowing it to be treated as a standard database within Python.
    *   **KPI Generation**: SQL queries were constructed and executed via the SQLAlchemy engine to retrieve data for three core KPIs:
        *   **Monthly Sales Trends (MoM Growth)**: Calculated `total_revenue` and `total_orders` grouped by month from `fact_orders`, also deriving month-over-month revenue growth using `pct_change()` in `pandas`.
        *   **Top-Selling Products (Revenue by Category)**: Joined `fact_orders` and `dim_product` to aggregate `category_revenue` and `items_sold` by product category, ordered by revenue.
        *   **Customer Segmentation (RFM Lite)**: Joined `fact_orders` and `dim_customer` to calculate customer `frequency` and `monetary_value`. A simple Python function (`segment_customer`) then classified customers into 'VIP / Loyal', 'High Value / New', and 'Standard' segments based on these metrics.
    *   **Visualization**: `matplotlib` and `seaborn` were used to create informative charts: line plots for sales trends, bar plots for top product categories, and pie charts for customer segmentation distribution. Charts were saved and made available for download.
*   **Key Tools**: `SQLAlchemy`, `sqlalchemy-bigquery`, `pandas`, `matplotlib`, `seaborn`.
*   **Outcome**: Actionable business insights were extracted, along with concrete suggestions for a new e-commerce venture based on observed trends in sales, product performance, and customer behavior.

## 2. Data Lineage and Architecture Documentation

This section details the end-to-end data flow and architectural design of the e-commerce data pipeline solution. It provides an overview of how raw data is transformed and organized for analytical consumption.

**Data Lineage Overview**

The detailed technical steps, data transformations, and tool usage for each stage of the data pipeline can be found in the **'Technical Approach Summary'** section above. This summary outlines the journey from initial data injection into DuckDB, through the creation of the `main_orders_view`, the design and population of the star schema in the `analytics` schema, and the subsequent ELT pipeline to Google BigQuery.

**Star Schema Architecture**

The relational structure of the star schema data warehouse, illustrating the relationships between the fact and dimension tables, is visually represented in the following diagram:

![Star Schema Relationship Map](https://github.com/chinwarsoon/dsai-5m-projects/blob/main/m2-project/star_schema_relationship_map.png?raw=1)

This diagram clearly shows the central `fact_orders` table connected to its surrounding dimension tables: `dim_customer`, `dim_product`, `dim_seller`, `dim_geolocation`, and `dim_time`. Each foreign key in the `fact_orders` table points to the primary key of its corresponding dimension table, facilitating efficient analytical queries by allowing data to be sliced and diced across various dimensions.

**Overall Data Flow Summary**

1.  **Raw Data Ingestion**: Raw CSV files are loaded into an in-memory DuckDB database, forming the initial 'Raw Schema'.
2.  **Data Consolidation**: A `main_orders_view` is created in DuckDB by joining several raw tables to provide a unified source of order-related data.
3.  **Persistence**: The in-memory DuckDB database, including the `main_orders_view`, is exported to a persistent `olist_ecommerce.db` file.
4.  **Star Schema Transformation**: Within the persistent DuckDB, a dedicated `analytics` schema is created, where dimension tables (`dim_customer`, `dim_product`, `dim_seller`, `dim_geolocation`, `dim_time`) and a `fact_orders` table are defined and populated. Surrogate keys and 'Unknown' entries are implemented to ensure data integrity.
5.  **ELT to BigQuery**: The curated star schema tables from DuckDB are extracted and loaded into Google BigQuery using `pandas_gbq`, establishing the final data warehouse.
6.  **Data Quality & Analytics**: Data quality checks are performed on the BigQuery data, and subsequently, an analytics engine uses SQLAlchemy to query the BigQuery star schema for KPI generation and visualization.

## 3. Synthesize Data Analysis Key Findings

**1. Monthly Sales Trends:**
*   The e-commerce platform demonstrates an **overall upward trend in both total revenue and total orders**, signifying a healthy and expanding market.
*   **Noticeable monthly fluctuations** suggest inherent seasonality in sales, with some periods experiencing faster growth or temporary plateaus. Revenue and order counts generally move in tandem, indicating a strong correlation between order volume and financial performance.

**2. Top-Selling Product Categories:**
*   **"cama_mesa_banho" (bed_bath_table), "beleza_saude" (health_beauty), and "informatica_acessorios" (computers_accessories)** are the top-performing categories, consistently generating the highest revenue.
*   These categories often represent household essentials, personal care items, and technology, implying sustained consumer demand across diverse product types.
*   The analysis highlights clear areas of existing high demand where strategic focus could yield significant returns.

**3. Customer Segmentation (RFM Lite):**
*   The customer base is predominantly composed of **"Standard Customers" (76.1%)**, who likely make one-off or low-value purchases. This large segment presents a significant opportunity for conversion into more loyal customers.
*   **"High Value / New Customers" (23.5%)** represent a valuable segment with high initial spending, indicating strong potential for repeat business with targeted engagement.
*   A smaller but highly impactful segment of **"VIP / Loyal Customers" (0.4%)** contributes significantly through frequent and high-value transactions, underscoring the importance of retention strategies for this group.

These findings collectively paint a picture of an active e-commerce market with identifiable growth patterns, strong product performers, and distinct customer behaviors that can be leveraged for strategic business decisions.

## 4. Justify Tool Choices

## 5. Final Report Outline



1. Introduction
    *   Project Overview
    *   Objective of the Data Pipeline and Analytics Engine
    *   Dataset Description (Olist E-commerce Data)

2. Technical Approach Summary
    *   Data Injection into DuckDB (Raw Schema)
    *   Creation of `main_orders_view`
    *   Export to a Persistent DuckDB File (`olist_ecommerce.db`)
    *   Star Schema Design and Population (in `analytics` schema)
    *   ELT Pipeline to Google BigQuery using `pandas_gbq`
    *   Data Quality Testing (Volumetry, Nulls, Duplicates, Referential Integrity Checks & Remediation)
    *   Analytics Engine using SQLAlchemy for KPI Generation

3. Data Lineage and Architecture
    *   Visual representation of the Star Schema (`star_schema_relationship_map.png`)
    *   Detailed Data Flow Description

4. Data Analysis Key Findings
    *   Monthly Sales Trends (MoM Growth)
    *   Top-Selling Product Categories (Revenue by Category)
    *   Customer Segmentation (RFM Lite)

5. Business Insights and Recommendations
    *   Actionable Product Strategy
    *   Target Market & Marketing Approach
    *   Operational & Growth Strategies
    *   Potential Future Strategies

6. Conclusion
    *   Summary of Achievements
    *   Future Work and Enhancements